# Strategy - Volume Compression

In [ ]:
# ============================================================
# S06 - Volatility Compression
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    close = xr.where(
        is_liquid == 1,
        close,
        np.nan
    )

    # --------------------------------------------------------
    # Daily Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    # --------------------------------------------------------
    # Volatility Measures
    # --------------------------------------------------------

    vol20 = qnta.std(ret, 20)

    vol60 = qnta.std(ret, 60)

    # --------------------------------------------------------
    # Compression Signal
    # --------------------------------------------------------

    compression = vol20 / (vol60 + 1e-6)

    # lower compression = better

    score = (-compression).rank("asset")

    # --------------------------------------------------------
    # Select Strongest Assets
    # --------------------------------------------------------

    ranks = score.rank("asset")

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Vol Weighting
    # --------------------------------------------------------

    inv_vol = 1 / (vol20 + 1e-6)

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    denom = weights.sum("asset")

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    return weights.fillna(0)

# ============================================================
# Generate Weights
# ============================================================

weights = strategy(data)

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field         equity  relative_return  volatility  underwater  max_drawdown  \
time                                                                          
2026-06-05  8.929158        -0.080545    0.804354   -0.946038     -0.956686   
2026-06-06  8.868507        -0.006792    0.804252   -0.946405     -0.956686   
2026-06-07  9.333904         0.052477    0.804301   -0.943592     -0.956686   
2026-06-08  9.229836        -0.011149    0.804205   -0.944221     -0.956686   
2026-06-09  9.116108        -0.012322    0.804111   -0.944908     -0.956686   

field       sharpe_ratio  mean_return  bias  instruments  avg_turnover  \
time                                                                     
2026-06-05      0.667598     0.536985   1.0         33.0      0.240700   
2026-06-06      0.666698     0.536193   1.0         33.0      0.240804   
2026-06-07      0.672731     0.541078   1.0         33.0      0.240763   
2026-06-08      0.671307     0.539869   1.0         33.0      0.240965   
2026-06-09      0.669743     0.538548   1.0         33.0      0.240928   

field       avg_holding_time  
time                          
2026-06-05          8.990169  
2026-06-06          8.996763  
2026-06-07          8.997012  
2026-06-08          8.995331  
2026-06-09          8.982466  

Final Metrics:
field
equity              9.116108
sharpe_ratio        0.669743
max_drawdown       -0.956686
avg_turnover        0.240928
avg_holding_time    8.982466
Name: 2026-06-09 00:00:00, dtype: float64